# Practice Lab: Controlled Generation (Lesson 6.3)

Module 6 · 8 exercises · ControlNet + LoRA + IP-Adapter
**Needs:** diffusers, controlnet-aux, torch

Exercises 7 (catalog) and 8 (ComfyUI JSON) run locally.


# Lesson 6.3: Controlled Generation — ControlNet, LoRA & IP-Adapter

8 exercises: 2E + 3M + 3C
**Needs:** `pip install diffusers transformers controlnet-aux torch`

Exercises 7 (catalog) and 8 (ComfyUI JSON) run **locally** (pure Python).


In [ ]:
!pip install diffusers transformers controlnet-aux accelerate torch -q


In [ ]:
import torch, os, csv, json, time
from PIL import Image, ImageDraw


---
## Exercise 7 (Challenge): Indian Product Catalog

CSV + Hindi text + metadata. Pure Python portion runs locally.


In [ ]:
# YOUR CODE
# TODO: create products list with Hindi names
# TODO: write CSV
# TODO: generate metadata JSON per product
# TODO: calculate cost savings


<details><summary>Solution (CSV + metadata, no GPU)</summary>


In [ ]:
import csv, json, os

products = [
    {'name': 'Banarasi Saree',
     'hindi': '\u092c\u0928\u093e\u0930\u0938\u0940 \u0938\u093e\u0921\u093c\u0940',
     'category': 'clothing',
     'style': 'elegant Banarasi silk saree, gold zari'},
    {'name': 'Brass Diya Set',
     'hindi': '\u092a\u0940\u0924\u0932 \u0926\u0940\u092f\u093e \u0938\u0947\u091f',
     'category': 'home_decor',
     'style': 'set of 5 brass oil lamps, ornate'},
    {'name': 'Kolhapuri Chappal',
     'hindi': '\u0915\u094b\u0932\u094d\u0939\u093e\u092a\u0941\u0930\u0940 \u091a\u092a\u094d\u092a\u0932',
     'category': 'footwear',
     'style': 'handcrafted leather Kolhapuri sandals'},
    {'name': 'Madhubani Painting',
     'hindi': '\u092e\u0927\u0941\u092c\u0928\u0940 \u092a\u0947\u0902\u091f\u093f\u0902\u0917',
     'category': 'art',
     'style': 'Madhubani Radha Krishna, vibrant'},
    {'name': 'Masala Dabba',
     'hindi': '\u092e\u0938\u093e\u0932\u093e \u0921\u092c\u094d\u092c\u093e',
     'category': 'kitchen',
     'style': 'steel spice box, 7 compartments'},
]

os.makedirs('catalog', exist_ok=True)

with open('catalog/products.csv', 'w', newline='',
          encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=[
        'name', 'hindi', 'category', 'style'])
    w.writeheader()
    w.writerows(products)
print(f'CSV: {len(products)} products written')

for i, prod in enumerate(products):
    prompt = (f"{prod['style']}, product photography, "
              f"white background, studio lighting, 8k")
    meta = {
        'product': prod['name'],
        'hindi': prod['hindi'],
        'category': prod['category'],
        'prompt': prompt,
        'seed': i * 100,
        'model': 'sdxl-1.0',
        'cost_inr': 15,
    }
    fname = f"catalog/{i:03d}_{prod['name'].replace(' ','_')}"
    with open(f'{fname}.json', 'w', encoding='utf-8') as mf:
        json.dump(meta, mf, indent=2, ensure_ascii=False)
    print(f'  [{i+1}/{len(products)}] '
          f'{prod["name"]} ({prod["hindi"]})')

# Cost analysis
ai_cost = len(products) * 15
trad_cost = len(products) * 5000
savings = (1 - ai_cost / trad_cost) * 100
print(f'\nCost: AI Rs {ai_cost} vs Traditional Rs {trad_cost}')
print(f'Savings: {savings:.0f}%')

# Verify
with open('catalog/products.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'\nCSV verified: {len(rows)} rows')
metas = [f for f in os.listdir('catalog') if f.endswith('.json')]
print(f'Metadata files: {len(metas)}')
for mf in sorted(metas):
    with open(f'catalog/{mf}', encoding='utf-8') as f:
        d = json.load(f)
    print(f'  {mf}: {d["product"]} | {d["hindi"]}'
          f' | Rs {d["cost_inr"]}')


</details>


---
## Exercise 8 (Challenge): Full ComfyUI Workflow

Build workflow JSON + batch automation. Pure Python.


In [ ]:
# YOUR CODE
# TODO: build workflow dict
# TODO: make_prompt_workflow function
# TODO: batch 5 prompts


<details><summary>Solution (JSON construction, no GPU)</summary>


In [ ]:
import json

workflow = {
    '1': {'class_type': 'CheckpointLoaderSimple',
          'inputs': {'ckpt_name': 'sd_xl_base_1.0.safetensors'}},
    '2': {'class_type': 'CLIPTextEncode',
          'inputs': {'text': 'PLACEHOLDER', 'clip': ['1', 1]}},
    '3': {'class_type': 'CLIPTextEncode',
          'inputs': {'text': 'blurry, low quality', 'clip': ['1', 1]}},
    '4': {'class_type': 'EmptyLatentImage',
          'inputs': {'width': 1024, 'height': 1024, 'batch_size': 1}},
    '5': {'class_type': 'KSampler',
          'inputs': {'seed': 42, 'steps': 25, 'cfg': 7.5,
                     'sampler_name': 'euler', 'scheduler': 'normal',
                     'denoise': 1.0, 'model': ['1', 0],
                     'positive': ['2', 0], 'negative': ['3', 0],
                     'latent_image': ['4', 0]}},
    '6': {'class_type': 'VAEDecode',
          'inputs': {'samples': ['5', 0], 'vae': ['1', 2]}},
    '7': {'class_type': 'SaveImage',
          'inputs': {'images': ['6', 0],
                     'filename_prefix': 'comfy_output'}},
}

with open('comfy_workflow.json', 'w') as f:
    json.dump(workflow, f, indent=2)

def make_prompt_workflow(prompt, seed=42):
    w = json.loads(json.dumps(workflow))
    w['2']['inputs']['text'] = prompt
    w['5']['inputs']['seed'] = seed
    return w

# Verify node connections
print('ComfyUI Workflow Nodes:')
for nid, node in sorted(workflow.items()):
    cls = node['class_type']
    inputs = node['inputs']
    refs = [f'{k}->{v}' for k, v in inputs.items()
            if isinstance(v, list)]
    print(f'  Node {nid}: {cls}')
    if refs:
        print(f'    Connections: {", ".join(refs)}')

# Verify flow: 1->2->5->6->7
flow = ['1:Checkpoint', '2:CLIP+', '3:CLIP-',
        '4:LatentImage', '5:KSampler',
        '6:VAEDecode', '7:Save']
print(f'\nFlow: {" -> ".join(flow)}')

# Batch
prompts = [
    'Rajasthani palace at sunset',
    'Spice market in Old Delhi',
    'Kerala backwaters with houseboats',
    'Taj Mahal morning mist reflection',
    'Holi celebration in Mathura',
]

print(f'\nBatch jobs ({len(prompts)}):')
for i, p in enumerate(prompts):
    w = make_prompt_workflow(p, seed=i*100)
    assert w['2']['inputs']['text'] == p
    assert w['5']['inputs']['seed'] == i*100
    print(f'  [{i+1}] seed={i*100}: {p}')

# Verify JSON roundtrip
with open('comfy_workflow.json') as f:
    loaded = json.load(f)
assert len(loaded) == len(workflow)
print(f'\nJSON roundtrip verified: {len(loaded)} nodes')
print(f'File size: {os.path.getsize("comfy_workflow.json")} bytes')


</details>


---
## Exercise 1 (Easy): ControlNet Scales
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 1: ControlNet Scales
# Needs GPU runtime.
pass


---
## Exercise 2 (Easy): Inpainting: Object Removal
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 2: Inpainting: Object Removal
# Needs GPU runtime.
pass


---
## Exercise 3 (Medium): Multi-ControlNet Architecture
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 3: Multi-ControlNet Architecture
# Needs GPU runtime.
pass


---
## Exercise 4 (Medium): IP-Adapter + ControlNet
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 4: IP-Adapter + ControlNet
# Needs GPU runtime.
pass


---
## Exercise 5 (Medium): Post-Processing Pipeline
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 5: Post-Processing Pipeline
# Needs GPU runtime.
pass


---
## Exercise 6 (Challenge): Regional Prompting
Needs GPU. See practice-lab-6_3.html for full code.


In [ ]:
# Exercise 6: Regional Prompting
# Needs GPU runtime.
pass
